# NLU Shared Task - Demo Solution 2

This notebook demonstrates how to generate final predictions using the Solution 2 Neural Meta-Learner Ensemble. It expects that the individual transformer cross-encoders have already produced their probability CSVs.

## Environment Setup
Install required dependencies.

In [ ]:
!pip install -q torch pandas scikit-learn gdown


In [ ]:
import pandas as pd
from pathlib import Path
import sys
import os

# Ensure src is in the python path
sys.path.append(os.path.abspath('..'))

## Setup: Download Required Model Predictions
Before running the ensemble inference, we need to download the base model predictions from Google Drive and set up the outputs directory. This only needs to run once.


In [ ]:
# Only run this cell if the data and models have not been downloaded yet
import gdown

output_dir = Path('../outputs/solution2')
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Created output directory: {output_dir}')

# Download model predictions from Google Drive
gdrive_folder_url = "https://drive.google.com/drive/folders/1saMnwl_u3_FZMiDiWzap-eyhmXymznFx?usp=sharing"
print(f'Downloading model predictions from Google Drive...')

gdown.download_folder(url=gdrive_folder_url, output=str(output_dir), quiet=False, use_cookies=False)
print(f'Downloaded model predictions to {output_dir}')

models_dir = Path('../models/solution2')
models_dir.mkdir(parents=True, exist_ok=True)
print(f'Created models directory: {models_dir}')

# Download trained ensemble models from Google Drive
gdrive_models_url = "https://drive.google.com/drive/folders/1arKIOSEAZxAz4P_-MotKRo0aFjFg-QGp?usp=sharing"
print(f'Downloading trained ensemble models from Google Drive...')

gdown.download_folder(url=gdrive_models_url, output=str(models_dir), quiet=False, use_cookies=False)
print(f'Downloaded trained models to {models_dir}')


## 1. Run Ensemble Inference
We use the predefined ensemble prediction script. This will load the base probabilities for the specified split (e.g. `dev` or `test`), pass them through the 4-layer fully connected meta-learner, and calibrate the outputs.

In [ ]:
from src.solution2.ensemble.predict_ensemble import predict_probs

split_name = 'test' # Change to 'test' when evaluating the final test set

print(f'Running Neural Meta-Ensemble on split: {split_name}...')
results_df = predict_probs(split=split_name)

print(f'Generated {len(results_df)} predictions.')

## 2. Format and Save Submission
We extract only the `prediction` column (0 or 1) as required by the shared task spec and save it.

In [ ]:
submission_df = results_df[['prediction']].copy()

# Preview output
display(submission_df.head())

# Save to disk
out_dir = Path('../outputs/solution2')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f'predictions_demo_{split_name}.csv'
submission_df.to_csv(out_path, index=False)
print(f'Saved predictions to {out_path}')